# Lab 2 Part 2: checkpointed LoRA

The first code cell creates a pinned environment and prints its commit, versions,
device and seed. It does not replace packages inside the running notebook kernel.
Windows, Linux, Colab and Kaggle paths are handled without a hard-coded `/content`
directory on a PC. First-time setup needs internet and several GB of free disk.

Use a fresh kernel and Run All. The default is a full run. `LAB2_SMOKE_ONLY=1`
is only for developer checks and is clearly printed; those outputs are not a
completed GPU experiment. Keep your real executed copy after the full run.
Run only one GPU training/generation notebook at a time. Setup is locked, but
two concurrent large models can still exceed the GPU's memory.

RTX 4050 profile: batch 1, accumulation 4, fp16 frozen UNet, gradient checkpointing, cached latents and text conditioning. No bitsandbytes or xformers dependency. The lab grading target remains a 16 GB GPU.

In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)
subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: SMOKE ONLY: no full training or model-backed evaluation


{
  "commit": "3b12f56",
  "working_tree_dirty": true,
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Preflight and real optimizer smoke test
A tiny randomly initialized UNet tests forward/backward, frozen base weights, checkpoint save, adapter export and interrupted-run resume. It does not count as the SD 1.5 training run.

In [2]:
if not SMOKE_ONLY:
    run_script("tools/preflight.py", "--require-cuda")
run_script("part2_diffusion/src/smoke_test.py", *([] if SMOKE_ONLY else ["--require-cuda"]))

Running smoke_test.py


Update 1/2, noise MSE 1.154229


{


  "rank": 2,


  "steps": 1,


  "final_loss": 1.1542287468910217,


... (45 outputs left out)

  "gpu_hours": null,


  "peak_allocated_mib": null


}


PASS: cpu LoRA training, frozen base, save/resume, adapter round-trip and tiny diffusion generation


## Prepare the actual dataset
Corrupt images, duplicates and empty captions are counted. The seed really controls the 90/10 split. Evaluation prompts are frozen from held-out captions before either training run.

In [3]:
P2 = ROOT / "part2_diffusion"
if SMOKE_ONLY:
    print("SKIPPED: full dataset download and conditioning cache (developer smoke mode).")
else:
    run_script("part2_diffusion/src/prepare_data.py", "--output-dir", P2 / "data/processed")
    run_script("part2_diffusion/src/generate_eval.py", "--freeze-prompts")

SKIPPED: full dataset download and conditioning cache (developer smoke mode).


## Train both ranks
Only one UNet is trained at a time. Complete checkpoints are resumed automatically. A finished compatible run is kept. Dataset and configuration mismatches stop rather than mixing experiments.

In [4]:
if SMOKE_ONLY:
    print("SKIPPED: both full SD 1.5 runs. The tiny-model test is not submitted training.")
else:
    for name, rank in [("run1", 4), ("run2", 8)]:
        run_script("part2_diffusion/src/train_lora.py", "--config", P2 / "config.yaml",
                   "--run", name, "--rank", rank, "--resume-latest")

SKIPPED: both full SD 1.5 runs. The tiny-model test is not submitted training.


## Generate and calculate metrics
All 120 images use identical settings. Missing adapters are errors, never silently replaced by the base model. The checkpoint safety checker stays enabled. Inference offloads components instead of loading the whole pipeline into VRAM. KID, CLIP and LPIPS are actually computed; human judgments still need real raters.

In [5]:
if SMOKE_ONLY:
    print("SKIPPED: real image generation, quality metrics and human judgments.")
else:
    run_script("part2_diffusion/src/generate_eval.py", "--config", P2 / "config.yaml")
    run_script("part2_diffusion/src/evaluate.py", "--config", P2 / "config.yaml")
    for name in ["run1", "run2"]:
        print(name, json.loads((P2 / "outputs" / name / "training_metrics.json").read_text()))
    print((P2 / "outputs/evaluation.json").read_text())
    from IPython.display import Image, display
    for index in range(8):
        display(Image(filename=str(P2 / "outputs/generations" / f"comparison_{index:02d}.png")))

SKIPPED: real image generation, quality metrics and human judgments.


## Measured comparison table and report
The report is generated from saved files. It stays clearly marked as a draft while human ratings, visual failure analysis or uploads are pending. It is a Part 2 section, not the final combined IEEE report for all three parts.

In [6]:
run_script("part2_diffusion/src/submission.py")
run_script("part2_diffusion/src/report_pdf.py")
from IPython.display import HTML, display
import html
table = json.loads((P2 / "outputs/training_table.json").read_text())
columns = ["model", "rank", "trainable_parameters", "learning_rate", "steps", "batch_size",
           "gradient_accumulation_steps", "peak_allocated_mib", "training_seconds", "gpu_hours", "final_loss"]
if table:
    display(HTML("<table><tr>" + "".join("<th>" + html.escape(c) + "</th>" for c in columns) + "</tr>" +
        "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(c, "not recorded"))) + "</td>" for c in columns) + "</tr>" for row in table) + "</table>"))
else:
    print("No full training measurements yet. Smoke-test losses are not submitted as SD 1.5 results.")
print("Part 2 PDF:", P2 / "output/pdf/Part2_Findings.pdf")

Running submission.py


Part 2 training comparison


model | rank | trainable | LR | steps | batch | accumulation | peak MiB | seconds | GPU-hours | loss


PASS: 500_plus_images_seeded_disjoint_split


PASS: 20_fixed_prompts_from_heldout


PENDING: run1_cuda_training


... (27 outputs left out)

PENDING: license_and_redistribution_reviewed


Created /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part2_diffusion/output/pdf/Part2_Findings.pdf


Report status: DRAFT with pending evidence


No full training measurements yet. Smoke-test losses are not submitted as SD 1.5 results.
Part 2 PDF: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part2_diffusion/output/pdf/Part2_Findings.pdf


## Human review, saved for later
Inspect the generated images, then copy `outputs/review_template.json` to `outputs/review.json`. Record the selected run and why, plus at least two failure modes with model, prompt index, seed, observation and explanation. This must describe real images.

The `outputs/human_run1` folder contains 25 blinded pairs. Keep the key away from raters. Two team members and at least one external person each judge every pair. Save their actual judgments to `ratings.json` as a JSON list. Each row has `rater_id`, `role` (`team_member` or `external`), `pair_id` (0-24), `domain_style` and `prompt_match` (each `A`, `B` or `tie`). If you select run2, run the next cell once to prepare its blinded set without retraining.

In [7]:
PREPARE_RUN2_HUMAN_PAIRS = False
if PREPARE_RUN2_HUMAN_PAIRS and not SMOKE_ONLY:
    run_script("part2_diffusion/src/evaluate.py", "--config", P2 / "config.yaml", "--final-run", "run2", "--human-only")
else:
    print("Human judgments remain pending. No ratings or failure observations were invented.")

Human judgments remain pending. No ratings or failure observations were invented.


## Hugging Face account and artifact upload
Create your own account at https://huggingface.co/join, verify the email and choose an available username. The PDF does not specify a username. Make a write-scoped token in account settings; never put it in a code cell or chat.

When ready, set `UPLOAD_TO_HF=True`, enter your verified username, and confirm that the educational redistribution is permitted. Both adapters and all 120 generated images will go to private repositories by default. Give the instructor access. This does not upload original training images or rater identities. Leave this cell disabled until you are ready.

In [8]:
UPLOAD_TO_HF = False
HF_NAMESPACE = ""  # Your verified username or team organization, not a guessed name.
RIGHTS_REVIEWED = False
if UPLOAD_TO_HF:
    if SMOKE_ONLY or not HF_NAMESPACE or not RIGHTS_REVIEWED:
        raise ValueError("Finish the full run, enter your real namespace and review redistribution rights first.")
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face write token (hidden): ")
    try:
        run_script("part2_diffusion/src/submission.py", "--publish", "--namespace", HF_NAMESPACE,
                   "--acknowledge-data-rights")
    finally:
        os.environ.pop("HF_TOKEN", None)
else:
    print("Hugging Face upload deferred. Signup, verified username and explicit rights review are still needed.")

Hugging Face upload deferred. Signup, verified username and explicit rights review are still needed.


## Final audit and refresh
This prints pending items honestly. Once ratings, review and uploads are added, rerun this cell to refresh the report without training again. Keep the executed notebook and outputs. The common lab submission also needs a combined IEEE report, team records, links and tags.

In [9]:
run_script("part2_diffusion/src/submission.py")
run_script("part2_diffusion/src/report_pdf.py")
audit = json.loads((P2 / "outputs/submission_audit.json").read_text())
print("Part 2 evidence ready:", audit["ready"])
print("Still pending:", audit["pending"])
print("Saved PDF:", P2 / "output/pdf/Part2_Findings.pdf")

Running submission.py


Part 2 training comparison


model | rank | trainable | LR | steps | batch | accumulation | peak MiB | seconds | GPU-hours | loss


PASS: 500_plus_images_seeded_disjoint_split


PASS: 20_fixed_prompts_from_heldout


PENDING: run1_cuda_training


... (27 outputs left out)

PENDING: license_and_redistribution_reviewed


Created /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part2_diffusion/output/pdf/Part2_Findings.pdf


Report status: DRAFT with pending evidence


Part 2 evidence ready: False
Still pending: ['run1_cuda_training', 'run2_cuda_training', '120_images_same_protocol', '8_comparison_grids', 'kid_clip_lpips_for_all_models', 'final_run_with_explanation', '2_visually_reviewed_failure_modes', '25_blind_pairs_3_real_raters', 'memorization_10_images_and_3_pairs', 'both_adapters_and_image_dataset_uploaded', 'team_details_recorded', 'license_and_redistribution_reviewed']
Saved PDF: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part2_diffusion/output/pdf/Part2_Findings.pdf
